In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
import time
from datetime import datetime, timedelta
import pandas as pd
import urllib.parse
import random
import json
import os

# 봇 탐지를 피하기 위해 user agent로 설정
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:138.0) Gecko/20100101 Firefox/138.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36',
]

options = Options()
options.add_argument(f'user-agent={random.choice(user_agents)}')
options.add_experimental_option('excludeSwitches', ['enable-automation'])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument('--disable-blink-features=AutomationControlled')

# Chrome 드라이버 자동 다운로드 및 설정
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# navigator.webdriver 플래그 제거
driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
    'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
})

# 변수 정의
start_date = '2025.04.01'
end_date = '2025.04.30'
query = 'SK텔레콤'

# 검색 키워드 인코딩
encoded_query = urllib.parse.quote(query)

# 주소에 들어갈 검색구간 처리
# 주소 원문 해당 구간 : so%3Ar%2Cp%3Afrom20220816to20220831 -> ':', ',' 처리를 위해 인코딩 필요
nso_value = f"so:r,p:from{start_date.replace('.', '')}to{end_date.replace('.', '')}"

# 변수 처리를 포함한 주소
url = f'https://search.naver.com/search.naver?ssc=tab.news.all&query={encoded_query}&sm=tab_opt&sort=2&photo=0&field=0&pd=3&ds={start_date}&de={end_date}&docid=&related=0&mynews=0&office_type=0&office_section_code=0&news_office_checked=&nso={urllib.parse.quote(nso_value)}&is_sug_officeid=0&office_category=&service_area=2'

# 임시 링크 파일 경로
temp_links_path = os.path.join(os.getcwd(), '_temp_links.json')

# 창 열기
driver.get(url)

In [ ]:
## 설정한 날짜구간에서 2000건이 넘으면 날짜 줄여서 다시 실행
## 또는 해당 날짜 기억 후 다음 기사 추출 시 그 날짜부터 실행 (통합 시 중복 제거 반드시 필요)

# 페이지 끝까지 내리기
# https://m.blog.naver.com/lmj4160/222462966573 에서 코드 발췌

# 현재 페이지 높이 = last_height
last_height = driver.execute_script("return document.body.scrollHeight")
PAUSE_SEC = 0.8 # 한 번에 끝까지 내리고 싶으면 1초 이상으로 설정하는 것을 권장

while True:
    # 스크롤 끝까지 내리기
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

    # 스크롤 내린 후 페이지 로딩을 위한 시간이 필요하다면, PAUSE_SEC에 숫자 입력
    time.sleep(PAUSE_SEC)

    # 스크롤 내린 후의 페이지 높이 = new_height
    new_height = driver.execute_script("return document.body.scrollHeight")

    # 더이상 스크롤이 내려가지 않으면 스크롤 내리는 반복문 멈춤 (=last_height와 new_height 같으면 멈춤)
    if new_height == last_height:
        break
    # 스크롤 내린 후의 페이지 높이(new_height)를 현재 페이지 높이(last_height) 변수에 저장
    last_height = new_height

In [ ]:
naver_news_links = []  # 링크들을 저장할 리스트

# 문서의 모든 <a> 태그를 찾되 href 속성에 n.news.naver.com이 포함되어 있는 <a>태그만 a_tags에 저장
a_tags = driver.find_elements(By.XPATH, '//a[contains(@href, "n.news.naver.com")]')
naver_news_links = [a.get_attribute('href') for a in a_tags] # a_tag에 저장된 <a>태그에서 href 속성 값(링크) 추출 후 저장

print(len(naver_news_links)) # 전체 기사 개수
print(naver_news_links[:5])

In [ ]:
# 수집한 링크를 임시 파일로 저장 (본문 수집 중 에러 시 링크 재수집 불필요)
with open(temp_links_path, 'w', encoding='utf-8') as f:
    json.dump(naver_news_links, f, ensure_ascii=False)

print(f"링크 {len(naver_news_links)}개 임시 저장 완료: {temp_links_path}")

In [ ]:
# 추출한 링크에 직접 방문하여 크롤링 진행 - 교수님 제공 코드 이용

# 크롤링 속도를 향상시키기 위해 time.sleep대신 WebDriverWait으로 변경 (다만 네트워크 불량 시 오래 걸릴 수 있음)
# WebDriverWait 이용 시 탐색 속도가 너무 빨라 봇 탐지 가능성 존재 -> time.sleep 이용 요망

# 진행상황 확인코드 추가

# 임시 저장된 링크가 있으면 불러오기 (링크 재수집 불필요)
if os.path.exists(temp_links_path):
    with open(temp_links_path, 'r', encoding='utf-8') as f:
        naver_news_links = json.load(f)
    print(f"임시 파일에서 링크 {len(naver_news_links)}개 불러옴")
else:
    print("임시 파일 없음 — 현재 메모리의 링크 사용")

all_results = dict()
i = 0
err_idx = []

for link in naver_news_links[i:]:
    try:
        all_results[i] = dict()

        # 실제 네이버 뉴스 웹페이지로 이동
        driver.get(link)

        # 페이지 로딩 대기
        time.sleep(0.6)

        # # 제목, 본문, 날짜가 로딩될 때까지 대기
        # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, 'media_end_head_headline')))
        # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'newsct_article')))
        # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME")))

        # 제목 추출하기
        title = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
        title = title[0].text 

        # 본문 추출하기
        body = driver.find_elements(By.ID, 'newsct_article')
        body = body[0].text.replace('\n', '') 

        # 날짜 추출하기
        pubdate_element = driver.find_elements(By.CSS_SELECTOR, "span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME")
        pubdate = pubdate_element[0].get_attribute("data-date-time") 

        all_results[i]['link'] = link
        all_results[i]['pubdate'] = pubdate
        all_results[i]['title'] = title
        all_results[i]['body'] = body

        # 진행 상황 확인용 코드
        print(f"[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% 진행")    
        i += 1

        # 봇 탐지 방지를 위해 0.5초에서 1.2초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.5, 1.2))

    # 오류 발생 시
    except:
        print('오류가 발생했습니다.')
        err_idx.append(i)

        print(f"[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% 진행 - 오류 발생")    
        i += 1

        # 봇 탐지 방지를 위해 0.5초에서 1.2초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.5, 1.2))

print(len(all_results))
print(err_idx)

In [ ]:
# err_idx가 빈 리스트가 아닐 때만 자동 재시도
# 전체 주석 설정 및 해제하고 싶으면 Ctrl+A로 전체 선택 후 Ctrl+/ 입력

if len(err_idx) != 0:
    print(f"\n오류 {len(err_idx)}건 재시도 시작...")
    re_err_idx = []

    for i in err_idx:
        try:
            link = naver_news_links[i]
            all_results[i] = dict()

            # 실제 네이버 뉴스 웹페이지로 이동
            driver.get(link)

            # 페이지 로딩 대기
            time.sleep(0.6)

            # # 제목, 본문, 날짜가 로딩될 때까지 대기
            # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, 'media_end_head_headline')))
            # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'newsct_article')))
            # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME")))

            # 제목 추출하기
            title = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
            title = title[0].text

            # 본문 추출하기
            body = driver.find_elements(By.ID, 'newsct_article')
            body = body[0].text.replace('\n', '')

            # 날짜 추출하기
            pubdate_element = driver.find_elements(By.CSS_SELECTOR, "span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME")
            pubdate = pubdate_element[0].get_attribute("data-date-time")

            all_results[i]['link'] = link
            all_results[i]['pubdate'] = pubdate
            all_results[i]['title'] = title
            all_results[i]['body'] = body

            # 진행 상황 확인용 코드
            print(f"[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% 재시도 성공")

            # 봇 탐지 방지를 위해 0.5초에서 1.2초 사이에 랜덤한 시간을 기다림
            time.sleep(random.uniform(0.5, 1.2))

        # 오류 발생 시
        except:
            re_err_idx.append(i)
            print(f"[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% 재시도 실패")
            time.sleep(random.uniform(0.5, 1.2))

    print(f"\n재시도 완료 — 성공: {len(err_idx) - len(re_err_idx)}건 / 재실패: {len(re_err_idx)}건")
    if re_err_idx:
        print(f"재실패 인덱스: {re_err_idx}")
else:
    print("오류 없음 — 재시도 불필요")

In [5]:
# 수집한 정보들을 dataframe으로 변환
df = pd.DataFrame(all_results).T
df

,link,pubdate,title,body
0,https://n.news.naver.com/mnews/article/005/000...,2025-04-01 00:22:27,부산시 “핀테크 키워 ‘디지털 금융 허브’로”,2028년까지 20개 유망기업 선정투자 생태계 조성·인재 양성 등단순 지원 넘어 성...
1,https://n.news.naver.com/mnews/article/023/000...,2025-04-01 00:32:27,"한때 발포제 세계 1위 금양, 이차전지 진출했다 상폐 위기",최근 6개년 사업보고서 보니31일 오전 부산 사상구에 있는 이차전지(배터리) 기업 ...
2,https://n.news.naver.com/mnews/article/018/000...,2025-04-01 05:41:09,식품은 이미 레드오션…서비스업 진출 잇따라,"[진화하는 펫코노미]②시몬스, 업계 최초 펫 매트리스 'N32 쪼꼬미' 출시펫 서비..."
3,https://n.news.naver.com/mnews/article/018/000...,2025-04-01 05:41:12,“펫산업도 진화중…시니어 반려동물 시장 유망”,"[진화하는 펫코노미]③유로모니터 문경선 총괄·방혜원 책임연구원 인터뷰“펫 산업, 밖..."
4,https://n.news.naver.com/mnews/article/001/001...,2025-04-01 05:50:00,"[샷!] ""탄핵 관련 거짓말만은""…만우절 경계경보","유쾌했던 만우절 → 불안정한 정국에 가짜뉴스 주의보""우리의 '가짜'에 대한 태도, ..."
...,...,...,...,...
231,https://n.news.naver.com/mnews/article/648/000...,2025-04-04 16:08:10,탄핵 선고된 11시22분…카톡도 멈췄다,트래픽 급증 여파…네이버는 안정적통신3사도 트래픽 폭증에 긴급대비윤석열 대통령 탄핵...
232,https://n.news.naver.com/mnews/article/009/000...,2025-04-04 17:31:18,"SKT ""멀티모달 AI 연내 개발""",유영상 대표 'AI전략 2.0' 공유AI 컴퍼니로 가시적 성과 창출SK텔레콤이 연내...
233,https://n.news.naver.com/mnews/article/015/000...,2025-04-04 18:07:16,"SKT ""돈버는 AI, 올해부터 성과낼 것""","'르네상스' 선언한 유영상 대표AI비서 순항·LLM 개발 마무리""수요는 확실…공급자..."
234,https://n.news.naver.com/mnews/article/014/000...,2025-04-04 18:36:22,"유영상 SKT 대표 ""구독형 GPU로 AI시장서 성과 낼것""","사내 메시지 통해 AI전략 공유연내 멀티모달·추론형 AI 개발""맞춤형 DC로 수익화..."


In [6]:
# 수집한 기사들 중 중복인 경우 이를 제거
df_no_duplicates = df.drop_duplicates().reset_index(drop=True)
df_no_duplicates

,link,pubdate,title,body
0,https://n.news.naver.com/mnews/article/005/000...,2025-04-01 00:22:27,부산시 “핀테크 키워 ‘디지털 금융 허브’로”,2028년까지 20개 유망기업 선정투자 생태계 조성·인재 양성 등단순 지원 넘어 성...
1,https://n.news.naver.com/mnews/article/023/000...,2025-04-01 00:32:27,"한때 발포제 세계 1위 금양, 이차전지 진출했다 상폐 위기",최근 6개년 사업보고서 보니31일 오전 부산 사상구에 있는 이차전지(배터리) 기업 ...
2,https://n.news.naver.com/mnews/article/018/000...,2025-04-01 05:41:09,식품은 이미 레드오션…서비스업 진출 잇따라,"[진화하는 펫코노미]②시몬스, 업계 최초 펫 매트리스 'N32 쪼꼬미' 출시펫 서비..."
3,https://n.news.naver.com/mnews/article/018/000...,2025-04-01 05:41:12,“펫산업도 진화중…시니어 반려동물 시장 유망”,"[진화하는 펫코노미]③유로모니터 문경선 총괄·방혜원 책임연구원 인터뷰“펫 산업, 밖..."
4,https://n.news.naver.com/mnews/article/001/001...,2025-04-01 05:50:00,"[샷!] ""탄핵 관련 거짓말만은""…만우절 경계경보","유쾌했던 만우절 → 불안정한 정국에 가짜뉴스 주의보""우리의 '가짜'에 대한 태도, ..."
...,...,...,...,...
228,https://n.news.naver.com/mnews/article/648/000...,2025-04-04 16:08:10,탄핵 선고된 11시22분…카톡도 멈췄다,트래픽 급증 여파…네이버는 안정적통신3사도 트래픽 폭증에 긴급대비윤석열 대통령 탄핵...
229,https://n.news.naver.com/mnews/article/009/000...,2025-04-04 17:31:18,"SKT ""멀티모달 AI 연내 개발""",유영상 대표 'AI전략 2.0' 공유AI 컴퍼니로 가시적 성과 창출SK텔레콤이 연내...
230,https://n.news.naver.com/mnews/article/015/000...,2025-04-04 18:07:16,"SKT ""돈버는 AI, 올해부터 성과낼 것""","'르네상스' 선언한 유영상 대표AI비서 순항·LLM 개발 마무리""수요는 확실…공급자..."
231,https://n.news.naver.com/mnews/article/014/000...,2025-04-04 18:36:22,"유영상 SKT 대표 ""구독형 GPU로 AI시장서 성과 낼것""","사내 메시지 통해 AI전략 공유연내 멀티모달·추론형 AI 개발""맞춤형 DC로 수익화..."


In [7]:
# 오래된 순부터 수집했으나 혹시 모를 상황을 방지하기 위해 pubdate를 datetime으로 변환 후 정렬
df_no_duplicates['pubdate'] = pd.to_datetime(df_no_duplicates['pubdate'])
df_sorted = df_no_duplicates.sort_values(by='pubdate')
df_sorted

,link,pubdate,title,body
0,https://n.news.naver.com/mnews/article/005/000...,2025-04-01 00:22:27,부산시 “핀테크 키워 ‘디지털 금융 허브’로”,2028년까지 20개 유망기업 선정투자 생태계 조성·인재 양성 등단순 지원 넘어 성...
1,https://n.news.naver.com/mnews/article/023/000...,2025-04-01 00:32:27,"한때 발포제 세계 1위 금양, 이차전지 진출했다 상폐 위기",최근 6개년 사업보고서 보니31일 오전 부산 사상구에 있는 이차전지(배터리) 기업 ...
2,https://n.news.naver.com/mnews/article/018/000...,2025-04-01 05:41:09,식품은 이미 레드오션…서비스업 진출 잇따라,"[진화하는 펫코노미]②시몬스, 업계 최초 펫 매트리스 'N32 쪼꼬미' 출시펫 서비..."
3,https://n.news.naver.com/mnews/article/018/000...,2025-04-01 05:41:12,“펫산업도 진화중…시니어 반려동물 시장 유망”,"[진화하는 펫코노미]③유로모니터 문경선 총괄·방혜원 책임연구원 인터뷰“펫 산업, 밖..."
4,https://n.news.naver.com/mnews/article/001/001...,2025-04-01 05:50:00,"[샷!] ""탄핵 관련 거짓말만은""…만우절 경계경보","유쾌했던 만우절 → 불안정한 정국에 가짜뉴스 주의보""우리의 '가짜'에 대한 태도, ..."
...,...,...,...,...
228,https://n.news.naver.com/mnews/article/648/000...,2025-04-04 16:08:10,탄핵 선고된 11시22분…카톡도 멈췄다,트래픽 급증 여파…네이버는 안정적통신3사도 트래픽 폭증에 긴급대비윤석열 대통령 탄핵...
229,https://n.news.naver.com/mnews/article/009/000...,2025-04-04 17:31:18,"SKT ""멀티모달 AI 연내 개발""",유영상 대표 'AI전략 2.0' 공유AI 컴퍼니로 가시적 성과 창출SK텔레콤이 연내...
230,https://n.news.naver.com/mnews/article/015/000...,2025-04-04 18:07:16,"SKT ""돈버는 AI, 올해부터 성과낼 것""","'르네상스' 선언한 유영상 대표AI비서 순항·LLM 개발 마무리""수요는 확실…공급자..."
232,https://n.news.naver.com/mnews/article/014/000...,2025-04-04 18:36:20,尹탄핵 순간 트래픽 폭주… 통신 마비는 없어,이통3사·네카오 선제적 대응카톡 8분가량 지연된 후 복구헌법재판소가 재판관 전원일치...


In [ ]:
import os

# 수집한 정보들을 csv로 저장 (이때 인코딩 형식은 utf8)
# 저장할 하위폴더 지정 (현재 작업 폴더 아래)
subfolder = 'data'  # 필요에 따라 변경. 빈 문자열로 하면 현재 폴더에 저장
save_dir = os.path.join(os.getcwd(), subfolder) if subfolder else os.getcwd()
os.makedirs(save_dir, exist_ok=True)

file_name = f"{query}_{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}.csv"
save_path = os.path.join(save_dir, file_name)

df_sorted.to_csv(save_path, index=False, encoding='utf-8-sig')

In [ ]:
# CSV 저장 완료 후 임시 링크 파일 삭제
if os.path.exists(temp_links_path):
    os.remove(temp_links_path)
    print(f"임시 파일 삭제 완료: {temp_links_path}")
else:
    print("임시 파일 없음 (이미 삭제되었거나 생성되지 않음)")